In [1]:
import os
from pathlib import Path

import numpy as np
import math
import uproot
import pickle
import pandas as pd

from XRootD import client
import tempfile

# Load dfs

In [2]:
def increase_first_idx(df, n):
    df.index = df.index.set_levels(
        df.index.levels[0] + n,
        level=0
    )
    return df

In [3]:
def merge_dfs(path, outdir, fname, n_files):
    dfs = {
        "mcnu": [],
        "hdr": [],
        "cuts": [],
        "pot": []
    }

    offset = 0

    for i in range(n_files):
        file = f"{fname}_{i}.df"

        # File nuevo por iteración evita cualquier estado colgado del anterior
        c = client.File()
        status, _ = c.open(path + file)
        if not status.ok:
            raise RuntimeError(f"Error abriendo {file}: {status.message}")

        status, data = c.read(0, 10**9)
        if not status.ok:
            raise RuntimeError(f"Error leyendo {file}: {status.message}")

        c.close()

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".df")
        tmp.write(data)
        tmp.close()

        with pd.HDFStore(tmp.name, 'r') as store:
            if not store.keys():
                continue
            
        try:
            mcnu = pd.read_hdf(tmp.name, key="mcnu_0")
            hdr = pd.read_hdf(tmp.name, key="hdr_0")
            cuts = pd.read_hdf(tmp.name, key="cuts_0")
            pot = pd.read_hdf(tmp.name, key="histpotdf_0")
        finally:
            os.remove(tmp.name)

        mcnu = increase_first_idx(mcnu, offset)
        hdr = increase_first_idx(hdr, offset)
        cuts = increase_first_idx(cuts, offset)
        pot = increase_first_idx(pot, offset)

        dfs["mcnu"].append(mcnu)
        dfs["hdr"].append(hdr)
        dfs["cuts"].append(cuts)
        dfs["pot"].append(pot)

        offset = pot.index.droplevel(-1).max() + 1

    merged = {k: pd.concat(v) for k, v in dfs.items()}

    with pd.HDFStore(outdir) as store:
        for k, df in merged.items():
            store.put(k, df)

In [4]:
outdir = "/exp/sbnd/data/users/syebes/eafData/dfs/lambda.df"
#path = "root://fndcadoor.fnal.gov//pnfs/fnal.gov/usr/sbnd/scratch/users/syebes/cafpyana_out/dfs/2026_06_24_085158__lambda/"
path = "root://fndcadoor.fnal.gov//pnfs/fnal.gov/usr/sbnd/scratch/users/syebes/cafpyana_out/dfs/2026_07_22_085952__lambda/"

fname = 'lambda'
n_files = 200

merge_dfs(path, outdir, fname, n_files)

/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.mc.nu..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.slc..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)


In [5]:
outdir = "/exp/sbnd/data/users/syebes/eafData/dfs/aurora_gen2.df"
#path = "root://fndcadoor.fnal.gov//pnfs/fnal.gov/usr/sbnd/scratch/users/syebes/cafpyana_out/dfs/2026_06_24_085016__gen2_aurora/"
path = "root://fndcadoor.fnal.gov//pnfs/fnal.gov/usr/sbnd/scratch/users/syebes/cafpyana_out/dfs/2026_07_22_100134__gen2_aurora/"
fname = 'gen2_aurora'
n_files = 500

merge_dfs(path, outdir, fname, n_files)

/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.mc.nu..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.slc..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)


In [67]:
outdir = "/exp/sbnd/data/users/syebes/eafData/dfs/lambda.df"
path = "root://fndcadoor.fnal.gov//pnfs/fnal.gov/usr/sbnd/scratch/users/syebes/cafpyana_out/dfs/2026_06_24_085158__lambda/"

ids = np.arange(0,500,1)
c = client.File()

dfs = {
    "mcnu": [],
    "hdr": [],
    "cuts": [],
    #"histgenevtdf": [],
    #"histpotdf": [],
    "pot": []
}

offset = 0

for i in ids:
    file = f"lambda_{i}.df"
    c.open(path + file)

    status, data = c.read(0, 10**9)

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".df")
    tmp.write(data)
    tmp.close()

    mcnu = pd.read_hdf(tmp.name, key="mcnu_0")
    hdr = pd.read_hdf(tmp.name, key="hdr_0")
    cuts = pd.read_hdf(tmp.name, key="cuts_0")
    #histgenevtdf = pd.read_hdf(tmp.name, key="histgenevtdf_0")
    #histpotdf = pd.read_hdf(tmp.name, key="histpotdf_0")
    pot = pd.read_hdf(tmp.name, key="histpotdf_0")

    # shift índices si hace falta
    mcnu = increase_first_idx(mcnu, offset)
    hdr = increase_first_idx(hdr, offset)
    cuts = increase_first_idx(cuts, offset)
    #histgenevtdf = increase_first_idx(histgenevtdf, offset)
    #histpotdf = increase_first_idx(histpotdf, offset)
    pot = increase_first_idx(pot, offset)

    dfs["mcnu"].append(mcnu)
    dfs["hdr"].append(hdr)
    dfs["cuts"].append(cuts)
    #dfs["histgenevtdf"].append(histgenevtdf)
    #dfs["histpotdf"].append(histpotdf)
    dfs["pot"].append(pot)

    offset = pot.index.droplevel(-1).max() + 1

    os.remove(tmp.name)

# concat final
merged = {k: pd.concat(v) for k, v in dfs.items()}

with pd.HDFStore(outdir) as store:
    for k, df in merged.items():
        store.put(k, df)

/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.mc.nu..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.slc..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)


In [ ]:
outdir = "/exp/sbnd/data/users/syebes/eafData/dfs/aurora_gen2.df"
path = "root://fndcadoor.fnal.gov//pnfs/fnal.gov/usr/sbnd/scratch/users/syebes/cafpyana_out/dfs/2026_06_24_085016__gen2_aurora/"

ids = np.arange(0,1000,1)
c = client.File()

dfs = {
    "mcnu": [],
    "hdr": [],
    "cuts": [],
    #"histgenevtdf": [],
    #"histpotdf": [],
    "pot": []
}

offset = 0

for i in ids:
    file = f"gen2_aurora_{i}.df"
    c.open(path + file)

    status, data = c.read(0, 10**9)

    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".df")
    tmp.write(data)
    tmp.close()

    mcnu = pd.read_hdf(tmp.name, key="mcnu_0")
    hdr = pd.read_hdf(tmp.name, key="hdr_0")
    cuts = pd.read_hdf(tmp.name, key="cuts_0")
    #histgenevtdf = pd.read_hdf(tmp.name, key="histgenevtdf_0")
    #histpotdf = pd.read_hdf(tmp.name, key="histpotdf_0")
    pot = pd.read_hdf(tmp.name, key="histpotdf_0")

    # shift índices si hace falta
    mcnu = increase_first_idx(mcnu, offset)
    hdr = increase_first_idx(hdr, offset)
    cuts = increase_first_idx(cuts, offset)
    #histgenevtdf = increase_first_idx(histgenevtdf, offset)
    #histpotdf = increase_first_idx(histpotdf, offset)
    pot = increase_first_idx(pot, offset)

    dfs["mcnu"].append(mcnu)
    dfs["hdr"].append(hdr)
    dfs["cuts"].append(cuts)
    #dfs["histgenevtdf"].append(histgenevtdf)
    #dfs["histpotdf"].append(histpotdf)
    dfs["pot"].append(pot)

    offset = pot.index.droplevel(-1).max() + 1

    os.remove(tmp.name)

# concat final
merged = {k: pd.concat(v) for k, v in dfs.items()}

with pd.HDFStore(outdir) as store:
    for k, df in merged.items():
        store.put(k, df)

/home/syebes/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/tables/attributeset.py:462: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.mc.nu..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
